# Lab 01-05 — Semantic chunking: draw chunk boundaries where meaning shifts

**Track 01 · Chunking** — compare *semantic* chunking against *recursive character* chunking on two real long-prose documents — the opening of `Pride and Prejudice` and of `Moby-Dick` from the Gutenberg corpus. Recursive character splitting cuts at a fixed character count, so a chunk can end in the middle of a sentence. A semantic splitter instead embeds every sentence and draws a chunk boundary exactly where the *meaning* shifts: where the embedding distance between two consecutive sentences jumps above the `breakpoint_percentile` percentile of all consecutive-sentence distances (here the 95th).

This notebook is **self-contained**: it imports sentence-transformers, numpy, and LangChain splitters directly — no repo component library. The semantic splitter is **hand-rolled right here** (BGE sentence embeddings + cosine jump detection) because `langchain-experimental` — where LangChain's `SemanticChunker` lives — is not installed and must not be imported. The inline class below is the same algorithm the shared `src/splitters/semantic.py` component runs.

```
raw prose -> inline Gutenberg stripper -> sentence split (regex) -> BGE embed every sentence
          -> cosine distance between consecutive sentences -> p95 threshold -> chunks
Data  : Data/corpus/gutenberg/ (pride-and-prejudice.txt, moby-dick.txt), first 50,000 chars each
Model : BAAI/bge-base-en-v1.5 (local BGE, CPU)
Compare: RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
```

What to look for in the output:

* the semantic splitter produces more, smaller, topically coherent chunks — roughly one per chapter of each novel;
* every boundary it draws sits at a sentence-level cosine-distance spike that clears the 95th-percentile threshold;
* the recursive splitter's 500-char chunks are blind to those topic boundaries and cut mid-sentence.

All embeddings are local (`BAAI/bge-base-en-v1.5` via sentence-transformers); the model downloads on first use into `~/.cache/huggingface`.


## Setup

One prerequisite must hold before this notebook will run:

- **gutenberg corpus on disk** — `Data/corpus/gutenberg/` with `pride-and-prejudice.txt` and `moby-dick.txt` (public-domain novels, already fetched by the repo's manifest-verified fetchers).

No repo imports are needed: everything this notebook uses comes from `sentence-transformers`, `numpy`, `langchain-core`, and `langchain-text-splitters`. The imports cell walks up to the repo root and cd's into it, because a notebook has no `__file__` — so every `Data/...` path resolves exactly like the lab script. Unlike the Curriculum notebook, there is no `sys.path` trick: nothing is imported from `src/`.

The next cell installs the notebook-specific dependencies (a no-op if you already ran `pip install -r requirements.txt`).


In [ ]:
# Lab-specific dependencies (already in requirements.txt — the install
# below is a no-op if you have run `pip install -r requirements.txt`).
%pip install -q sentence-transformers numpy langchain-core langchain-text-splitters


In [ ]:
# Bootstrap: stdlib imports + repo-root walk (no sys.path tricks).
from __future__ import annotations

import os
import re
from pathlib import Path

# numpy + LangChain splitters — the only libraries this notebook needs.
# Nothing is imported from the repo's src/ component library.
import numpy as np  # noqa: E402
from langchain_core.documents import Document  # noqa: E402
from langchain_text_splitters import RecursiveCharacterTextSplitter  # noqa: E402

# A notebook has no __file__, so walk up from the cwd to the repo root and
# cd into it — Data/... paths then resolve exactly like the lab script.
REPO_ROOT = Path.cwd()
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "src" / "curriculum").is_dir() and (_candidate / "NoteBooks").is_dir():
        REPO_ROOT = _candidate
        break
os.chdir(REPO_ROOT)


## 1. Configuration

`MODEL_NAME` is the local BGE model (the same default the repo's semantic splitter uses); `BREAKPOINT_PERCENTILE` is the distance percentile above which a boundary is drawn; `CHUNK_SIZE`/`CHUNK_OVERLAP` are the recursive splitter's budget; `SUBSET_CHARS` bounds each book to its first 50K characters — semantic splitting embeds every sentence locally, so a full ~700KB novel would be far too slow (50K characters of prose is roughly 1,300 sentences and keeps one run in the low minutes). `DISTANCE_PRINT_CAP` caps the printed distance array, and the two preview caps keep the demo readable.


In [ ]:
# ---------------------------------------------------------------------------
# 1. Configuration — tweak these to rerun the experiment
# ---------------------------------------------------------------------------
MODEL_NAME = "BAAI/bge-base-en-v1.5"  # local BGE model (splitters/semantic.py default)
BREAKPOINT_PERCENTILE = 95.0  # distance percentile above which a boundary is drawn
CHUNK_SIZE = 500  # recursive splitter: max characters per chunk
CHUNK_OVERLAP = 50  # recursive splitter: characters shared across chunks
SUBSET_CHARS = 50_000  # per-book prefix cap (keeps sentence embedding runtime sane)
DISTANCE_PRINT_CAP = 40  # max consecutive-sentence distances printed per doc
DOC_PATHS = [
    Path("Data/corpus/gutenberg/pride-and-prejudice.txt"),
    Path("Data/corpus/gutenberg/moby-dick.txt"),
]
PREVIEW = 200  # character cap for chunk previews
SENTENCE_PREVIEW = 80  # character cap for boundary-neighbour sentences


## 2. Load — Gutenberg novels, boilerplate stripped, bounded prefix

The repo's `GutenbergLoader` strips the Project Gutenberg header/footer; the inline version below does the same marker slice. Each book's text is then sliced to the first `SUBSET_CHARS` characters so the sentence-by-sentence BGE embedding stays within a sane runtime, and wrapped as one `Document` tagged with its source path.


In [ ]:
# ---------------------------------------------------------------------------
# 2. Load — Gutenberg novels -> Documents with source metadata (bounded prefix)
# ---------------------------------------------------------------------------
START_MARKER = "*** START OF THE PROJECT GUTENBERG EBOOK"
END_MARKER = "*** END OF THE PROJECT GUTENBERG EBOOK"


def strip_gutenberg_boilerplate(text: str) -> str:
    """Slice from just after the START marker to just before the END marker."""
    start = text.find(START_MARKER)
    end = text.find(END_MARKER)
    if start == -1 or end == -1 or end < start:
        return text.strip()
    return text[start + len(START_MARKER):end].strip()


def load_books(paths: list[Path]) -> list[Document]:
    """Load Gutenberg novels, boilerplate stripped, each bounded to a prefix.

    Each book's text is sliced to the first ``SUBSET_CHARS`` characters so
    the sentence-by-sentence BGE embedding stays within a sane runtime.
    """
    docs: list[Document] = []
    for path in paths:
        text = strip_gutenberg_boilerplate(path.read_text(encoding="utf-8"))
        docs.append(
            Document(
                page_content=text[:SUBSET_CHARS],
                metadata={"source": str(path)},
            )
        )
    return docs


## 3. Semantic splitter — hand-rolled inline

`langchain-experimental` (where LangChain's `SemanticChunker` lives) is not installed and must not be imported, so the semantic splitter is built right here, matching `src/splitters/semantic.py` behavior exactly:

1. split each Document into sentences (a light regex split on `.`/`?`/`!` followed by whitespace);
2. embed every sentence with a sentence-transformer model;
3. compute pairwise cosine distances between consecutive embeddings;
4. find the `breakpoint_percentile` percentile of those distances; every distance above it marks a chunk boundary;
5. join sentences between breakpoints into one chunk Document, carrying over the source metadata.

The class accepts any embedding provider exposing `embed_documents`, `embed_query`, or `encode` — the lab passes the raw `SentenceTransformer` model, so the splitter and the boundary analysis below operate on identical sentence embeddings.


In [ ]:
# ---------------------------------------------------------------------------
# 3. Semantic splitter — hand-rolled inline (BGE + cosine jump detection)
# ---------------------------------------------------------------------------
# Sentence-boundary split: a period/question mark/exclamation point followed
# by whitespace (or end of string). Kept deliberately simple — no external
# sentence-tokenizer dependency.
_SENTENCE_RE = re.compile(r"(?<=[.!?])\s+")


class SemanticSplitter:
    """Split documents into semantically coherent chunks.

    Chunks group consecutive sentences whose embeddings stay close to each
    other; a chunk boundary is inserted where the embedding distance between
    two neighbouring sentences exceeds ``breakpoint_percentile`` percent of
    all such distances. Same algorithm as ``src/splitters/semantic.py``.
    """

    def __init__(self, embedding=None, breakpoint_percentile: float = 95.0):
        self.embedding = embedding
        self.breakpoint_percentile = breakpoint_percentile

    def _get_encoder(self):
        """Return a callable ``encoder(sentences) -> numpy.ndarray``."""
        if self.embedding is not None:
            return self._embed_with_provider
        import sentence_transformers  # lazy import

        model = sentence_transformers.SentenceTransformer("BAAI/bge-base-en-v1.5")
        return lambda sentences: model.encode(sentences)

    def _embed_with_provider(self, sentences: list[str]) -> np.ndarray:
        """Embed sentences using the user-supplied ``embedding`` provider."""
        if hasattr(self.embedding, "embed_documents"):
            vectors = self.embedding.embed_documents(sentences)
            return np.asarray(vectors, dtype="float32")
        if hasattr(self.embedding, "embed_query"):
            return np.asarray(
                [self.embedding.embed_query(sentence) for sentence in sentences],
                dtype="float32",
            )
        if hasattr(self.embedding, "encode"):
            return np.asarray(self.embedding.encode(sentences), dtype="float32")
        raise TypeError(
            "SemanticSplitter.embedding must provide embed_documents(), "
            "embed_query(), or encode(); got "
            f"{type(self.embedding).__name__}"
        )

    def _split_one(self, doc: Document, encoder) -> list[Document]:
        """Split a single Document into semantic chunks."""
        sentences = _SENTENCE_RE.split(doc.page_content.strip())
        sentences = [s for s in sentences if s]
        if not sentences:
            return []

        # Short text: a single sentence never spans a breakpoint, so it is
        # already one chunk — no need to embed anything.
        if len(sentences) == 1:
            return [Document(page_content=sentences[0], metadata=dict(doc.metadata))]

        embeddings = np.asarray(encoder(sentences), dtype="float32")
        norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
        norms[norms == 0] = 1.0  # guard against zero-vector sentences
        normalized = embeddings / norms

        # Cosine distance between consecutive sentences = 1 - cosine similarity.
        dots = np.sum(normalized[:-1] * normalized[1:], axis=1)
        distances = 1.0 - np.clip(dots, -1.0, 1.0)

        threshold = float(np.percentile(distances, self.breakpoint_percentile))
        breakpoints = [i for i, d in enumerate(distances) if d > threshold]

        # Build chunks: sentence indices are split at the breakpoints
        # (a breakpoint at i means the boundary is *after* sentence i).
        boundaries = [0] + [b + 1 for b in breakpoints] + [len(sentences)]
        chunks: list[Document] = []
        for start, end in zip(boundaries, boundaries[1:]):
            joined = " ".join(sentences[start:end])
            chunks.append(
                Document(page_content=joined, metadata=dict(doc.metadata))
            )
        return chunks

    def split_docs(self, docs: list[Document]) -> list[Document]:
        """Split documents into semantic chunks (canonical entry point)."""
        encoder = self._get_encoder()
        chunks: list[Document] = []
        for doc in docs:
            chunks.extend(self._split_one(doc, encoder))
        return chunks

    def split(self, docs: list[Document]) -> list[Document]:
        """Split documents into semantic chunks (alias of ``split_docs``)."""
        return self.split_docs(docs)


## 4. Boundary inspection — the teaching point

`consecutive_distances` computes the cosine distance between consecutive sentences (1 - cosine similarity), mirroring the internals of the splitter above so the numbers printed here are exactly the ones the `SemanticSplitter` used to draw boundaries. `boundary_pairs` returns the first `limit` boundaries as `(pair_no, index, distance, prev, next)` tuples, and `preview` collapses whitespace and caps a sentence/chunk preview.


In [ ]:
# ---------------------------------------------------------------------------
# 4. Boundary inspection — the teaching point
# ---------------------------------------------------------------------------
def consecutive_distances(model, sentences: list[str]) -> np.ndarray:
    """Cosine distance between consecutive sentences (1 - cosine similarity).

    Mirrors the internals of ``src/splitters/semantic.py`` so the numbers
    printed here are exactly the ones the SemanticSplitter used to draw
    boundaries.
    """
    vectors = np.asarray(model.encode(sentences), dtype="float32")
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    norms[norms == 0] = 1.0  # guard against zero-vector sentences
    normalized = vectors / norms
    dots = np.sum(normalized[:-1] * normalized[1:], axis=1)
    return 1.0 - np.clip(dots, -1.0, 1.0)


def boundary_pairs(
    sentences: list[str], distances: np.ndarray, threshold: float, limit: int = 3
) -> list[tuple[int, int, float, str, str]]:
    """First ``limit`` boundaries as (pair_no, index, distance, prev, next)."""
    pairs: list[tuple[int, int, float, str, str]] = []
    boundary_count = 0
    for i, distance in enumerate(distances):
        if distance > threshold:
            boundary_count += 1
            pairs.append(
                (boundary_count, i, float(distance), sentences[i], sentences[i + 1])
            )
            if len(pairs) == limit:
                break
    return pairs


def preview(text: str, limit: int) -> str:
    """Collapse whitespace and cap a sentence/chunk preview at ``limit`` chars."""
    flat = " ".join(text.split())
    return flat if len(flat) <= limit else flat[:limit] + "..."


## 5. Comparison helpers

`avg_words` reports the average word count across a chunk list — the size signal that shows the semantic splitter's chunks are smaller and topically coherent while the recursive splitter's are fixed-size.


In [ ]:
# ---------------------------------------------------------------------------
# 5. Comparison helpers
# ---------------------------------------------------------------------------
def avg_words(chunks: list[Document]) -> float:
    """Average word count across chunks (0.0 for an empty chunk list)."""
    if not chunks:
        return 0.0
    return sum(len(c.page_content.split()) for c in chunks) / len(chunks)


## 6. Experiment — split, inspect boundaries, compare

`run_experiment` runs the full protocol: load the two bounded books; build ONE local BGE model shared by the splitter and the boundary analysis (so both operate on identical sentence embeddings); split with the inline `SemanticSplitter` and with `RecursiveCharacterTextSplitter(500/50)`; compare chunk counts and average sizes overall and per book; then, per book, recompute the consecutive-sentence distances, the p95 threshold, the marked distance array (every boundary the splitter drew flagged with `*`), and the first boundary pairs with their neighbours — the spike pattern is the teaching point. Finally it captures the content previews of one semantic chunk and one recursive chunk from the same prose excerpt.

The heavy step is the sentence-by-sentence BGE embedding (~1,300 sentences per book); the cell caps torch threads so the run stays light on shared machines — the threshold and boundaries are percentile-based, so nothing measured changes.


In [ ]:
# ---------------------------------------------------------------------------
# 6. Experiment — setup -> load -> split -> inspect boundaries -> compare
# ---------------------------------------------------------------------------
def run_experiment() -> dict:
    # Keep the embedding run light on shared machines (results are
    # percentile-based, so the measured quantities do not change).
    try:
        import torch

        torch.set_num_threads(2)
    except ImportError:
        pass

    docs = load_books(DOC_PATHS)

    try:
        import sentence_transformers
    except ImportError:
        print(
            "SKIP: semantic chunking needs sentence-transformers: "
            "pip install sentence-transformers"
        )
        return {"skipped": True}

    # One local BGE model shared by the splitter and the boundary analysis,
    # so both operate on identical sentence embeddings.
    model = sentence_transformers.SentenceTransformer(MODEL_NAME)
    semantic = SemanticSplitter(
        embedding=model, breakpoint_percentile=BREAKPOINT_PERCENTILE
    )
    recursive = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP
    )

    semantic_chunks = semantic.split(docs)
    recursive_chunks = recursive.split_documents(docs)

    # --- Teaching point: boundaries sit at cosine-distance spikes -----------
    per_doc = []
    for doc in docs:
        sentences = [
            s for s in _SENTENCE_RE.split(doc.page_content.strip()) if s
        ]
        if len(sentences) < 2:
            per_doc.append(
                {"source": doc.metadata["source"], "sentences": sentences,
                 "distances": None, "threshold": None, "pairs": []}
            )
            continue
        distances = consecutive_distances(model, sentences)
        threshold = float(np.percentile(distances, BREAKPOINT_PERCENTILE))
        per_doc.append(
            {
                "source": doc.metadata["source"],
                "sentences": sentences,
                "distances": distances,
                "threshold": threshold,
                "pairs": boundary_pairs(sentences, distances, threshold),
            }
        )

    # --- Content preview: semantic chunks group related sentences -----------
    first_semantic = (
        semantic_chunks[1] if len(semantic_chunks) > 1 else semantic_chunks[0]
    )
    first_recursive = (
        recursive_chunks[1] if len(recursive_chunks) > 1 else recursive_chunks[0]
    )

    return {
        "docs": docs,
        "semantic_chunks": semantic_chunks,
        "recursive_chunks": recursive_chunks,
        "per_doc": per_doc,
        "first_semantic": first_semantic,
        "first_recursive": first_recursive,
    }


## 7. Demo — print the artifact

`print_demo(exp)` prints the artifact in the lab's order: the loaded books with their bounded sizes; the chunk-count and average-size comparison for both splitters, overall and per book; the per-book boundary analysis — sentence counts, distance stats (mean/max/p95 threshold), the marked distance array (first `DISTANCE_PRINT_CAP` distances, `*` = boundary), and the first boundary pairs with their neighbouring sentences; the content previews of one semantic chunk vs one recursive chunk from the same prose; and the teaching takeaway.


In [ ]:
# ---------------------------------------------------------------------------
# 7. Demo — print the artifact
# ---------------------------------------------------------------------------
def print_demo(exp: dict) -> None:
    if exp.get("skipped"):
        print(
            "SKIP: semantic chunking needs sentence-transformers: "
            "pip install sentence-transformers"
        )
        return

    docs = exp["docs"]
    semantic_chunks = exp["semantic_chunks"]
    recursive_chunks = exp["recursive_chunks"]
    per_doc = exp["per_doc"]

    print(
        f"Loaded {len(docs)} Gutenberg books (each limited to the first "
        f"{SUBSET_CHARS} characters):"
    )
    for doc in docs:
        print(f"  {doc.metadata['source']} — {len(doc.page_content)} chars")

    # --- Compare: chunk counts + average size -------------------------------
    print(
        f"Recursive split (chunk_size={CHUNK_SIZE}, chunk_overlap={CHUNK_OVERLAP}): "
        f"{len(recursive_chunks)} chunk(s), avg {avg_words(recursive_chunks):.1f} words/chunk"
    )
    print(
        f"Semantic split (breakpoint_percentile={BREAKPOINT_PERCENTILE:.0f}): "
        f"{len(semantic_chunks)} chunk(s), avg {avg_words(semantic_chunks):.1f} words/chunk"
    )
    for doc in docs:
        source = doc.metadata["source"]
        n_recursive = sum(
            1 for c in recursive_chunks if c.metadata.get("source") == source
        )
        n_semantic = sum(1 for c in semantic_chunks if c.metadata.get("source") == source)
        print(f"  {source}: {n_recursive} recursive chunk(s) vs {n_semantic} semantic chunk(s)")

    # --- Teaching point: boundaries sit at cosine-distance spikes -----------
    print("\n--- Where does the semantic splitter draw boundaries? ---")
    for entry in per_doc:
        if entry["distances"] is None:
            print(f"{entry['source']}: too few sentences to analyse")
            continue
        distances = entry["distances"]
        threshold = entry["threshold"]
        sentences = entry["sentences"]
        print(
            f"\n{entry['source']} — {len(sentences)} sentence(s), "
            f"{len(distances)} consecutive-sentence distances"
        )
        print(
            f"  distance stats: mean={distances.mean():.3f}  "
            f"max={distances.max():.3f}  "
            f"p{BREAKPOINT_PERCENTILE:.0f} threshold={threshold:.3f}"
        )
        # The distance array with '*' marking every boundary the splitter
        # drew: the spike pattern is visible at a glance. Capped at
        # DISTANCE_PRINT_CAP so a ~1300-sentence book doesn't flood output.
        shown = distances[:DISTANCE_PRINT_CAP]
        marked = " ".join(
            f"{d:.2f}" + ("*" if d > threshold else "") for d in shown
        )
        n_more = len(distances) - len(shown)
        more_note = f" ... ({n_more} more)" if n_more > 0 else ""
        print(
            f"  distances (first {len(shown)} of {len(distances)}): "
            f"{marked}{more_note}   (* = boundary, distance > p95 threshold)"
        )
        for pair_no, i, distance, prev_sentence, next_sentence in entry["pairs"]:
            print(
                f"  chunk pair {pair_no} -> {pair_no + 1}: boundary after sentence "
                f"{i + 1}, distance {distance:.3f} > p95 threshold {threshold:.3f} "
                f"(spike +{distance - threshold:.3f}, "
                f"{distance / distances.mean():.2f}x the mean distance)"
            )
            print(f"    ends: {preview(prev_sentence, SENTENCE_PREVIEW)}")
            print(f"    next: {preview(next_sentence, SENTENCE_PREVIEW)}")

    # --- Content preview: semantic chunks group related sentences -----------
    print("\n--- Content preview: semantic chunks group related sentences ---")
    # Same prose excerpt, two splitters: the semantic chunk keeps a stretch
    # of narrative (a chapter opening, a train of thought) whole because its
    # sentences embed closely; the recursive chunk cuts at a fixed 500
    # characters and can land mid-sentence.
    first_semantic = exp["first_semantic"]
    first_recursive = exp["first_recursive"]
    print(
        f"Semantic chunk [1] ({avg_words([first_semantic]):.0f} words, "
        f"source {first_semantic.metadata['source']}):"
    )
    print(f"  {preview(first_semantic.page_content, PREVIEW)}")
    print(
        f"Recursive chunk [1] ({avg_words([first_recursive]):.0f} words, "
        f"source {first_recursive.metadata['source']}):"
    )
    print(f"  {preview(first_recursive.page_content, PREVIEW)}")

    print(
        "\nTeaching takeaway: the semantic splitter draws a boundary at every "
        "sentence pair whose\ncosine distance clears the "
        f"{BREAKPOINT_PERCENTILE:.0f}th-percentile threshold — exactly where "
        "the topic shifts —\nwhile the recursive splitter cuts at a fixed "
        f"{CHUNK_SIZE} characters, blind to meaning."
    )


## 8. Verification gate

`verify_gate(exp)` enforces the lab's hard checks: both books loaded (bounded to `SUBSET_CHARS`); both splitters produced chunks for every book; the inline splitter is internally consistent — per book, the number of semantic chunks equals the number of drawn breakpoints plus one (every boundary the splitter drew sits above the p95 threshold, and every chunk boundary corresponds to one); and the threshold is a real cut — at least one consecutive-sentence distance clears it. Every check should print PASS.


In [ ]:
# ---------------------------------------------------------------------------
# 8. Verification gate
# ---------------------------------------------------------------------------
def verify_gate(exp: dict) -> int:
    checks: list[tuple[str, bool]] = []

    if exp.get("skipped"):
        print("verification gate:")
        print("  [SKIP] sentence-transformers not installed — experiment skipped")
        return 0

    docs = exp["docs"]
    semantic_chunks = exp["semantic_chunks"]
    recursive_chunks = exp["recursive_chunks"]
    per_doc = exp["per_doc"]

    checks.append((
        f"{len(DOC_PATHS)} Gutenberg books loaded, each bounded to "
        f"{SUBSET_CHARS} chars",
        len(docs) == len(DOC_PATHS)
        and all(len(d.page_content) <= SUBSET_CHARS for d in docs),
    ))
    for entry in per_doc:
        source = entry["source"]
        n_semantic = sum(
            1 for c in semantic_chunks if c.metadata.get("source") == source
        )
        n_recursive = sum(
            1 for c in recursive_chunks if c.metadata.get("source") == source
        )
        checks.append((
            f"semantic splitter produced chunks: {source}",
            n_semantic > 0,
        ))
        checks.append((
            f"recursive splitter produced chunks: {source}",
            n_recursive > 0,
        ))
        if entry["distances"] is None:
            continue
        # Internal consistency of the inline splitter: every chunk boundary
        # corresponds to one breakpoint (distance > threshold), and vice versa.
        n_breakpoints = int((entry["distances"] > entry["threshold"]).sum())
        checks.append((
            f"inline splitter consistent: {source} has {n_semantic} chunks == "
            f"{n_breakpoints} breakpoints + 1",
            n_semantic == n_breakpoints + 1,
        ))
        checks.append((
            f"threshold is a real cut — some distance clears p95: {source}",
            entry["distances"].max() > entry["threshold"],
        ))

    print("verification gate:")
    for label, ok in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
    return 0 if all(ok for _, ok in checks) else 1


## Run the experiment

A few minutes: ~2,600 sentence embeddings with local BGE on CPU (the model is cached) — no downloads, no API calls. `exp` holds everything the demo and gate need.


In [ ]:
exp = run_experiment()


### Demo — the artifact

The boundary spike pattern: where the semantic splitter drew its cuts (distance > p95 threshold) vs the fixed 500-char cuts of the recursive splitter — plus per-book chunk counts and the preview of one semantic chunk next to one recursive chunk from the same prose.


In [ ]:
print_demo(exp)


### Verification gate

Expect every check to PASS — the same gate the CI-style `--verify` run enforces. If any line shows FAIL, check the gutenberg files are intact.


In [ ]:
verify_gate(exp)
